In [ ]:
import spatrio
import pandas as pd
import argparse


def str2bool(v):
    if isinstance(v, bool):
        return v
    if v.lower() in ('yes', 'true', 't', 'y', '1'):
        return True
    elif v.lower() in ('no', 'false', 'f', 'n', '0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')




In [ ]:
import pandas as pd

# 读取 TSV 文件
df = pd.read_csv(
    '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/sc_expression.tsv',
    sep='\t',index_col=0
)

# 保存为 CSV
df.to_csv(
    '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/multi_rna.csv'
)
print("转换完成！")

In [21]:
df = pd.read_csv(
    '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/st_expression.tsv',
    sep='\t',index_col=0
)

# 保存为 CSV
df.to_csv(
    '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/spatial_rna.csv'
)
print("转换完成！")

转换完成！


In [29]:
import pandas as pd
df=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/spatialDWLS/Spot_cluster_zztry.txt', sep='\t')
df.index=df['cell_ID']
df.columns=['id','type','type1']
df[['id','type']].to_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/spatial_meta.csv',sep=',',header=True)

In [ ]:
df=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/cell_type_labels.tsv', sep='\t')
df.index=df['Cell IDs']
df.columns=['id','type']
df.to_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/multi_meta.csv', sep=',', header=True)

In [20]:
df=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/spatial_coords.tsv',index_col=0, sep='\t')
df.columns=['x', 'y']
df.to_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/pos.csv', sep=',', header=True)

In [ ]:

path = '/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio'
expected_num_path = None
top_num = None

print('\n','***Spatrio is running***','\n')

print('\n','***STEP1 LOAD DATA***','\n')
print('Loading data...')
spot_ann = spatrio.load_data(path+'/spatial_rna.csv' )
single_ann = spatrio.load_data( path+'/multi_rna.csv' )
spot_meta =  pd.read_csv(path+'/spatial_meta.csv', index_col=0)
spot_meta['type'] = spot_meta['type'].apply(lambda x: str(x))
single_meta =  pd.read_csv(path+'/multi_meta.csv', index_col=0)
single_meta['type'] = single_meta['type'].apply(lambda x: str(x))
spot_meta[['sample']] = 'spot'
single_meta[['sample']] = 'single'
spot_ann.obs['type'] = spot_meta['type']
spot_ann.obs['type'] = spot_ann.obs['type'].astype(object)
single_ann.obs['type'] = single_meta['type']
single_ann.obs['type'] = single_ann.obs['type'].astype(object)
print('Done!')

pos = pd.read_csv(path+'/pos.csv', index_col=0)
emb = pd.read_csv(path+'/emb.csv', index_col=0)
pos.index=spot_ann.obs_names
spot_ann.obsm['spatial'] = pos
emb.index=single_ann.obs_names
single_ann.obsm['reduction'] = emb

if expected_num_path is not None:
    expected_num = pd.read_csv(expected_num_path,index_col=0)
elif top_num is not None:
    expected_num = None


print('\n','***STEP2 PROCESS DATA***','\n')
print('Processing data...')
data1,data2 = spatrio.process_input(spot_ann,single_ann,
                                    marker_use = True,
                                    top_marker_num = 100,
                                    hvg_use = False)



expected_num = None

print('Done!')

print('\n','***STEP3 OT ALIGNMENT***','\n')
if expected_num is not None:
    expected_num = expected_num.loc[data1.obs_names]
    tmp_num = expected_num['cell_num'].values
    p_distribution  = tmp_num/sum(tmp_num)
else:
    p_distribution = None




print(f"Spots: {data1.n_obs}, Cells: {data2.n_obs}")


In [ ]:
spatrio_decon = spatrio.ot_alignment(adata1 = data1, 
                                     adata2 = data2, 
                                     dissimilarity = 'scaled_euc',
                                     alpha = 0.1, 
                                     k = 5,
                                     graph_mode = 'connectivity',
                                     aware_spatial = True,
                                     aware_multi = True,
                                     aware_power = 2,
                                     p_distribution = p_distribution)
print('\n','***STEP4 ASSIGN COORD***','\n')
spatrio_map = spatrio.assign_coord(adata1 = data1,
                                   adata2 = data2,
                                   out_data = spatrio_decon,
                                   top_num = 5,
                                   random = False,
                                   expected_num = expected_num)
output_path='/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/'
print('\n','The output was saved as '+str(output_path)+'/output.csv')
spatrio_map.to_csv(output_path+'/'+'output.csv')